# **YOLO11l-seg training (6-class eye segmentation)**

このノートブックでは、CVAT XMLファイルからYOLO形式のセグメンテーションデータセットを構築し、
YOLO11l-segを事前学習済み重みでトレーニングし、raw vs fullmaxマスク後処理によるサンプル推論を実行します。

## クラス定義

このノートブックでは **conj（結膜）** と **caruncle（涙丘）** を別クラスとして扱います。

| YOLO class_id (train/infer) | Class Name | 説明 |
|----------|------------|------|
| 0 | conj | 結膜 |
| 1 | caruncle | 涙丘 |
| 2 | iris_vis | 虹彩（可視部分） |
| 3 | iris_occ | 虹彩（遮蔽部分） |
| 4 | pupil_vis | 瞳孔（可視部分） |
| 5 | pupil_occ | 瞳孔（遮蔽部分） |

※ background（クラス0）はYOLOでは通常アノテーション不要のため、ラベル生成時にスキップします。

## ワークフロー

1. **データの準備**: CVAT XMLファイルから画像リストを取得
2. **YOLO形式への変換**: XMLからYOLO形式のセグメンテーションアノテーションを生成
3. **YOLO11l-segトレーニング**: YOLO11l-segモデルのトレーニング（回転augmentation: 0-180度）
4. **推論と可視化**: トレーニング済みモデルによる推論と結果の可視化

## 注意事項

- 入力画像リストは2つのCVAT XMLファイル（ID 0-2999範囲）から取得されます
- このノートブックは評価CSVファイルを生成しません
- YOLOはセグメンテーション形式（polygon）のアノテーションを使用します

In [22]:
from pathlib import Path
import json
import random
import xml.etree.ElementTree as ET
import math

import cv2
import numpy as np
import albumentations as A  # ブラー・低解像度augmentation用

PROJECT_ROOT = Path('.').resolve()
IMAGES_DIR = PROJECT_ROOT / 'Images' / 'images'
LABEL_SEG_DIR = PROJECT_ROOT / 'Images' / 'labels_seg'

XML_EYELID = PROJECT_ROOT / 'Images' / 'eyelid_caruncle_seg_0-3000.xml'
XML_IRIS_PUPIL = PROJECT_ROOT / 'Images' / 'obb_iris_pupil_1-3000.xml'

OUTPUT_DIR = PROJECT_ROOT / 'YOLO11l-seg_3000mai'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 512
SEED = 42
VAL_RATIO = 0.2
MIN_CONTOUR_AREA = 10
NUM_POINTS = 100  # YOLO形式の輪郭点の数

# YOLOは0-indexedのクラスIDを期待するため、0から始める
CLASS_NAMES = {
    0: 'conj',
    1: 'caruncle',
    2: 'iris_vis',
    3: 'iris_occ',
    4: 'pupil_vis',
    5: 'pupil_occ',
}

# YOLOラベルファイルの出力先
# 【重要】YOLOは画像パスから自動的にラベルパスを推測します
# 画像が Images/images/xxx.jpg の場合、ラベルは Images/labels/xxx.txt を探します
YOLO_LABELS_DIR = Path("Images/labels")
YOLO_LABELS_DIR.mkdir(parents=True, exist_ok=True)

print('OK: config loaded')

OK: config loaded


In [23]:
def load_cvat_image_names_with_id(xml_path: Path, min_id: int = 0, max_id: int = 2999):
    """
    XMLファイルから画像名とIDを取得し、指定されたID範囲でフィルタリング
    Returns: dict {image_id: image_name}
    """
    if not xml_path.exists():
        raise FileNotFoundError(xml_path)
    root = ET.parse(xml_path).getroot()
    image_dict = {}
    for img in root.findall('.//image'):
        img_id_str = img.attrib.get('id')
        name = img.attrib.get('name')
        if img_id_str is not None and name:
            try:
                img_id = int(img_id_str)
                if min_id <= img_id <= max_id:
                    image_dict[img_id] = Path(name).name
            except ValueError:
                continue
    return image_dict

# 画像ID 0-2999の範囲で画像を取得
eyelid_dict = load_cvat_image_names_with_id(XML_EYELID, min_id=0, max_id=2999)
iris_dict = load_cvat_image_names_with_id(XML_IRIS_PUPIL, min_id=0, max_id=2999)

# 和集合（union）を使用：両方のXMLファイルに含まれる全ての画像を使用
all_dict = {**eyelid_dict, **iris_dict}  # 後から来たもので上書き（同じIDの場合）

# ID順にソート
all_names = [all_dict[img_id] for img_id in sorted(all_dict.keys())]

image_names = []
missing_images = 0

# 画像が存在するか確認
for name in all_names:
    img_path = IMAGES_DIR / name
    if not img_path.exists():
        missing_images += 1
        continue
    image_names.append(name)

print(f'XML eyelid images (ID 0-2999): {len(eyelid_dict)}')
print(f'XML iris/pupil images (ID 0-2999): {len(iris_dict)}')
print(f'Union (all names, ID 0-2999): {len(all_dict)}')
print(f'Total images (ID 0-2999): {len(image_names)}')
print(f'Missing images: {missing_images}')

# train/val分割（患者単位）
def subject_id_from_name(name: str) -> str:
    return str(name).split('-', 1)[0]

subjects = sorted({subject_id_from_name(n) for n in image_names})
rng = random.Random(SEED)
rng.shuffle(subjects)

num_val = max(1, int(len(subjects) * VAL_RATIO))
val_subjects = set(subjects[:num_val])

train_names = [n for n in image_names if subject_id_from_name(n) not in val_subjects]
val_names = [n for n in image_names if subject_id_from_name(n) in val_subjects]

print(f'\nTrain images: {len(train_names)}')
print(f'Val images: {len(val_names)}')
print(f'Train subjects: {len({subject_id_from_name(n) for n in train_names})}')
print(f'Val subjects: {len(val_subjects)}')


XML eyelid images (ID 0-2999): 2990
XML iris/pupil images (ID 0-2999): 3000
Union (all names, ID 0-2999): 3000
Total images (ID 0-2999): 3000
Missing images: 0

Train images: 2426
Val images: 574
Train subjects: 156
Val subjects: 38


In [24]:
# ===== XML -> YOLO形式ラベル生成 =====

# XMLラベルからクラスIDへのマッピング
XML_LABEL_TO_CLASS_ID = {
    "Eyelid": 0,  # conj
    "Caruncle": 1,
    # Iris/Pupilは後でvis/occに分割
    "Iris": None,
    "Pupil": None,
}

# vis/occ分割ルール
EYELID_INSIDE_IS_VISIBLE = True

def _parse_points_attr(points_str: str):
    """CVAT polygon points="x,y;x,y;..." -> [x0,y0,x1,y1,...]"""
    pts = []
    for token in points_str.strip().split(";"):
        token = token.strip()
        if not token:
            continue
        x_str, y_str = token.split(",")
        pts.append(float(x_str))
        pts.append(float(y_str))
    return pts

def _ellipse_to_polygon(cx: float, cy: float, rx: float, ry: float, rotation_deg: float, n: int = 100, w: int | None = None, h: int | None = None):
    """楕円パラメータ -> polygon (x0,y0,...)"""
    theta = math.radians(rotation_deg % 360.0)
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)

    poly = []
    for i in range(n):
        a = 2.0 * math.pi * (i / n)
        ca = math.cos(a)
        sa = math.sin(a)

        x0 = rx * ca
        y0 = ry * sa
        x = cx + x0 * cos_t - y0 * sin_t
        y = cy + x0 * sin_t + y0 * cos_t

        if w is not None:
            x = max(0.0, min(float(w - 1), x))
        if h is not None:
            y = max(0.0, min(float(h - 1), y))

        poly.append(float(x))
        poly.append(float(y))

    return poly

def load_cvat_polygons_by_image(xml_path: Path):
    """CVAT XML(polygon) -> {image_name: [(label, poly), ...]}"""
    out = {}
    if not xml_path.exists():
        return out

    tree = ET.parse(str(xml_path))
    root = tree.getroot()
    for img in root.findall("image"):
        name = img.get("name")
        if not name:
            continue
        items = []
        for poly in img.findall("polygon"):
            label = poly.get("label")
            pts = poly.get("points")
            if not label or not pts:
                continue
            items.append((label, _parse_points_attr(pts)))
        if items:
            out[name] = items
    return out

def load_cvat_ellipses_by_image(xml_path: Path):
    """CVAT XML(ellipse) -> {image_name: [(label, cx,cy,rx,ry,rot_deg), ...]}"""
    out = {}
    if not xml_path.exists():
        return out

    tree = ET.parse(str(xml_path))
    root = tree.getroot()
    for img in root.findall("image"):
        name = img.get("name")
        if not name:
            continue
        items = []
        for el in img.findall("ellipse"):
            label = el.get("label")
            if not label:
                continue
            try:
                cx = float(el.get("cx"))
                cy = float(el.get("cy"))
                rx = float(el.get("rx"))
                ry = float(el.get("ry"))
            except Exception:
                continue
            rot = float(el.get("rotation")) if el.get("rotation") is not None else 0.0
            items.append((label, cx, cy, rx, ry, rot))
        if items:
            out[name] = items
    return out

# XMLを一度だけ読み込んでキャッシュ
EYELID_POLYS_BY_IMAGE = load_cvat_polygons_by_image(XML_EYELID)
ELLIPSES_BY_IMAGE = load_cvat_ellipses_by_image(XML_IRIS_PUPIL)

print("✓ XML読み込み完了")


✓ XML読み込み完了


In [25]:
# ===== 輪郭抽出・サンプリング関数 =====

def extract_contours_from_mask(mask, min_area=10):
    """マスクから輪郭を抽出"""
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    
    filtered_contours = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area >= min_area:
            filtered_contours.append(cnt)
    
    return filtered_contours

def sample_points_along_contour(contour, num_points=100):
    """輪郭に沿って均等に点をサンプリング"""
    if len(contour) < 2:
        return None
    
    contour_1d = contour.reshape(-1, 2).astype(np.float32)
    
    distances = np.zeros(len(contour_1d))
    for i in range(1, len(contour_1d)):
        dist = np.linalg.norm(contour_1d[i] - contour_1d[i-1])
        distances[i] = distances[i-1] + dist
    
    total_length = distances[-1]
    if total_length > 0:
        last_to_first = np.linalg.norm(contour_1d[0] - contour_1d[-1])
        if last_to_first < 10:
            total_length += last_to_first
    
    if total_length == 0:
        return None
    
    sampled_points = []
    for i in range(num_points):
        target_dist = (total_length * i) / num_points
        idx = np.searchsorted(distances, target_dist, side='right')
        idx = min(idx, len(contour_1d) - 1)
        
        if idx == 0:
            point = contour_1d[0]
        elif idx >= len(distances):
            point = contour_1d[-1]
        else:
            dist_before = distances[idx - 1]
            dist_after = distances[idx]
            if dist_after > dist_before:
                alpha = (target_dist - dist_before) / (dist_after - dist_before)
                point = contour_1d[idx - 1] * (1 - alpha) + contour_1d[idx] * alpha
            else:
                point = contour_1d[idx]
        
        sampled_points.append(point)
    
    return np.array(sampled_points, dtype=np.float32)

def polygon_to_yolo_line(poly: list[float], class_id: int, image_size: int = IMAGE_SIZE, num_points: int = NUM_POINTS):
    """polygonをYOLO形式の行に変換"""
    if len(poly) < 6:
        return None
    
    # polygonをnumpy配列に変換
    pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
    
    # 輪郭として扱う（閉じた輪郭）
    contour = pts.reshape(-1, 1, 2).astype(np.int32)
    
    # 100点サンプリング
    sampled_points = sample_points_along_contour(contour, num_points)
    if sampled_points is None or len(sampled_points) == 0:
        return None
    
    # 正規化座標に変換（0-1範囲）
    normalized_points = sampled_points / image_size
    
    # YOLO形式の行を作成
    yolo_line = f"{class_id}"
    for x, y in normalized_points:
        yolo_line += f" {x:.6f} {y:.6f}"
    
    return yolo_line

print("✓ 輪郭抽出・サンプリング関数を定義しました")


✓ 輪郭抽出・サンプリング関数を定義しました


In [26]:
# ===== XMLからYOLO形式ラベルを生成 =====

from tqdm import tqdm

def build_yolo_labels_from_xml(names):
    """XMLからYOLO形式のラベルファイルを生成"""
    stats = {
        'total': 0,
        'success': 0,
        'failed': 0,
        'class_counts': {i: 0 for i in range(6)}  # 0-5
    }
    
    for name in tqdm(names, desc="Generating YOLO labels"):
        stats['total'] += 1
        
        img_path = IMAGES_DIR / name
        if not img_path.exists():
            stats['failed'] += 1
            continue
        
        img = cv2.imread(str(img_path))
        if img is None:
            stats['failed'] += 1
            continue
        
        h, w = img.shape[:2]
        stem = Path(name).stem
        yolo_label_path = YOLO_LABELS_DIR / f"{stem}.txt"
        
        yolo_lines = []
        
        # polygon: Eyelid/Caruncle
        for xml_label, poly in EYELID_POLYS_BY_IMAGE.get(name, []):
            if xml_label not in XML_LABEL_TO_CLASS_ID:
                continue
            cid = int(XML_LABEL_TO_CLASS_ID[xml_label])
            if cid not in CLASS_NAMES:
                continue
            
            yolo_line = polygon_to_yolo_line(poly, cid, image_size=w, num_points=NUM_POINTS)
            if yolo_line is not None:
                yolo_lines.append(yolo_line)
                stats['class_counts'][cid] += 1
        
        # Iris/Pupil: 楕円をeyelid領域との交差でvis/occに分割
        eyelid_mask = np.zeros((h, w), dtype=np.uint8)
        for _lbl, _poly in EYELID_POLYS_BY_IMAGE.get(name, []):
            if _lbl != 'Eyelid':
                continue
            pts = np.array(_poly, dtype=np.float32).reshape(-1, 2)
            pts_i = np.round(pts).astype(np.int32)
            cv2.fillPoly(eyelid_mask, [pts_i], 255)
        
        for xml_label, cx, cy, rx, ry, rot in ELLIPSES_BY_IMAGE.get(name, []):
            if xml_label not in ('Iris', 'Pupil'):
                continue
            
            full_poly = _ellipse_to_polygon(cx, cy, rx, ry, rot, n=NUM_POINTS, w=w, h=h)
            
            ell_mask = np.zeros((h, w), dtype=np.uint8)
            pts = np.array(full_poly, dtype=np.float32).reshape(-1, 2)
            pts_i = np.round(pts).astype(np.int32)
            cv2.fillPoly(ell_mask, [pts_i], 255)
            
            inside = cv2.bitwise_and(ell_mask, eyelid_mask)
            outside = cv2.bitwise_and(ell_mask, cv2.bitwise_not(eyelid_mask))
            
            if EYELID_INSIDE_IS_VISIBLE:
                vis_mask = inside
                occ_mask = outside
            else:
                vis_mask = outside
                occ_mask = inside
            
            if xml_label == 'Iris':
                vis_cid, occ_cid = 2, 3
            else:  # Pupil
                vis_cid, occ_cid = 4, 5
            
            # vis_maskから輪郭を抽出してYOLO形式に変換
            vis_contours = extract_contours_from_mask(vis_mask, min_area=MIN_CONTOUR_AREA)
            for cnt in vis_contours:
                sampled_points = sample_points_along_contour(cnt, NUM_POINTS)
                if sampled_points is not None and len(sampled_points) > 0:
                    normalized_points = sampled_points / w
                    yolo_line = f"{vis_cid}"
                    for x, y in normalized_points:
                        yolo_line += f" {x:.6f} {y:.6f}"
                    yolo_lines.append(yolo_line)
                    stats['class_counts'][vis_cid] += 1
            
            # occ_maskから輪郭を抽出してYOLO形式に変換
            occ_contours = extract_contours_from_mask(occ_mask, min_area=MIN_CONTOUR_AREA)
            for cnt in occ_contours:
                sampled_points = sample_points_along_contour(cnt, NUM_POINTS)
                if sampled_points is not None and len(sampled_points) > 0:
                    normalized_points = sampled_points / w
                    yolo_line = f"{occ_cid}"
                    for x, y in normalized_points:
                        yolo_line += f" {x:.6f} {y:.6f}"
                    yolo_lines.append(yolo_line)
                    stats['class_counts'][occ_cid] += 1
        
        # YOLOラベルファイルを保存
        if len(yolo_lines) > 0:
            with open(yolo_label_path, 'w') as f:
                f.write('\n'.join(yolo_lines) + '\n')
            stats['success'] += 1
        else:
            # アノテーションが空の場合は空ファイルを作成
            yolo_label_path.touch()
            stats['success'] += 1
    
    return stats

# 全画像に対してYOLOラベルを生成
print("YOLOラベルを生成中...")
all_stats = build_yolo_labels_from_xml(image_names)

print("\n" + "=" * 80)
print("✅ YOLOラベル生成完了")
print("=" * 80)
print(f"総画像数: {all_stats['total']}")
print(f"成功: {all_stats['success']}")
print(f"失敗: {all_stats['failed']}")
print(f"\nクラス別アノテーション数:")
for class_id, count in all_stats['class_counts'].items():
    if count > 0:
        print(f"  クラス{class_id} ({CLASS_NAMES[class_id]}): {count}個")


YOLOラベルを生成中...


Generating YOLO labels: 100%|██████████| 3000/3000 [00:54<00:00, 54.68it/s] 


✅ YOLOラベル生成完了
総画像数: 3000
成功: 3000
失敗: 0

クラス別アノテーション数:
  クラス0 (conj): 2997個
  クラス1 (caruncle): 2567個
  クラス2 (iris_vis): 2967個
  クラス3 (iris_occ): 4470個
  クラス4 (pupil_vis): 2951個
  クラス5 (pupil_occ): 559個


In [27]:
# ===== YOLOデータセット設定ファイル生成 =====

# 画像パスリストファイルを作成
train_list_file = OUTPUT_DIR / 'yolo11_train_3000mai.txt'
val_list_file = OUTPUT_DIR / 'yolo11_val_3000mai.txt'

# train画像パスリスト
with open(train_list_file, 'w', encoding='utf-8') as f:
    for name in train_names:
        img_path = IMAGES_DIR / name
        f.write(f"{img_path.resolve()}\n")

# val画像パスリスト
with open(val_list_file, 'w', encoding='utf-8') as f:
    for name in val_names:
        img_path = IMAGES_DIR / name
        f.write(f"{img_path.resolve()}\n")

print(f"✓ 画像パスリストを作成しました")
print(f"  - Train: {train_list_file} ({len(train_names)} images)")
print(f"  - Val: {val_list_file} ({len(val_names)} images)")

# YOLO用YAMLファイルを作成
dataset_yaml_content = f"""# YOLO11データセット設定 (3000mai)
path: {OUTPUT_DIR.absolute()}
train: {train_list_file.absolute()}
val: {val_list_file.absolute()}

# クラス名
names:
  0: conj
  1: caruncle
  2: iris_vis
  3: iris_occ
  4: pupil_vis
  5: pupil_occ

# クラス数
nc: {len(CLASS_NAMES)}
"""

yaml_file = OUTPUT_DIR / 'dataset_yolo11_3000mai.yaml'
with open(yaml_file, 'w', encoding='utf-8') as f:
    f.write(dataset_yaml_content)

print(f"✓ YOLOデータセット設定ファイルを作成しました")
print(f"  - {yaml_file}")


✓ 画像パスリストを作成しました
  - Train: C:\Users\CorneAI\Eyelid_Iris_pupil_seg_comparison\YOLO11l-seg_3000mai\yolo11_train_3000mai.txt (2426 images)
  - Val: C:\Users\CorneAI\Eyelid_Iris_pupil_seg_comparison\YOLO11l-seg_3000mai\yolo11_val_3000mai.txt (574 images)
✓ YOLOデータセット設定ファイルを作成しました
  - C:\Users\CorneAI\Eyelid_Iris_pupil_seg_comparison\YOLO11l-seg_3000mai\dataset_yolo11_3000mai.yaml


In [28]:
# ===== YOLO11l-segトレーニング（ブラー・低解像度対策強化版） =====

RUN_TRAIN = True  # トレーニングを実行する場合は True に設定

# RUN_TRAINがFalseの場合は実行しない
if not RUN_TRAIN:
    print("トレーニングを実行するには RUN_TRAIN=True に設定してください。")
    if 'OUTPUT_DIR' in globals():
        print(f"\n💡 ヒント:")
        print(f"  - runsの保存場所: {OUTPUT_DIR / 'yolo11l-seg_3000mai' if 'OUTPUT_DIR' in globals() else 'YOLO11l-seg_3000mai/yolo11l-seg_3000mai'}")
        print(f"  - チェックポイント: {OUTPUT_DIR / 'yolo11l-seg_3000mai' / 'weights' / 'best.pt' if 'OUTPUT_DIR' in globals() else 'YOLO11l-seg_3000mai/yolo11l-seg_3000mai/weights/best.pt'}")
    else:
        print(f"\n💡 ヒント:")
        print(f"  - runsの保存場所: YOLO11l-seg_3000mai/yolo11l-seg_3000mai")
        print(f"  - チェックポイント: YOLO11l-seg_3000mai/yolo11l-seg_3000mai/weights/best.pt")
else:
    import torch
    from ultralytics import YOLO
    import cv2
    import albumentations as A

    # GPU確認
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"✓ GPUが利用可能")
        print(f"  - GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠ GPUが利用できません。CPUで実行します。")
        device = torch.device('cpu')

    # トレーニング設定
    EPOCHS = 100
    BATCH_SIZE = 8
    PATIENCE = 20  # early stopping
    YOLO_MODEL_NAME = "yolo11l-seg.pt"  # large版

    # ===== カスタムAugmentation（ブラー・低解像度対策） =====
    custom_transforms = [
        # ブラー系augmentation（30%の確率で適用）
        A.OneOf([
            A.MotionBlur(blur_limit=(3, 15), p=1.0),      # モーションブラー
            A.GaussianBlur(blur_limit=(3, 11), p=1.0),    # ガウシアンブラー
            A.MedianBlur(blur_limit=7, p=1.0),            # メディアンブラー
            A.Defocus(radius=(3, 10), alias_blur=(0.1, 0.5), p=1.0),  # ピンボケ
        ], p=0.3),
        
        # 低解像度シミュレーション（20%の確率で適用）
        A.OneOf([
            A.Downscale(scale_min=0.25, scale_max=0.5, interpolation=cv2.INTER_LINEAR, p=1.0),
            A.Downscale(scale_min=0.5, scale_max=0.75, interpolation=cv2.INTER_LINEAR, p=1.0),
        ], p=0.2),
        
        # ノイズaugmentation（15%の確率で適用）
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),  # ガウシアンノイズ
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),  # ISOノイズ
        ], p=0.15),
        
        # 画質劣化augmentation（10%の確率で適用）
        A.ImageCompression(quality_lower=50, quality_upper=90, p=0.1),  # JPEG圧縮アーティファクト
        
        # コントラスト強調（低コントラスト画像対策）
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.1),
    ]

    # runsの保存場所
    RUNS_DIR = OUTPUT_DIR / 'yolo11l-seg_3000mai'
    print(f"\n📁 runsの保存場所: {RUNS_DIR.absolute()}")
    print(f"  - チェックポイント: {RUNS_DIR / 'weights'}")
    print(f"  - 学習曲線など: {RUNS_DIR}")

    if RUN_TRAIN:
        print(f"\nトレーニング設定:")
        print(f"  - モデル: {YOLO_MODEL_NAME}")
        print(f"  - エポック数: {EPOCHS}")
        print(f"  - バッチサイズ: {BATCH_SIZE}")
        print(f"  - Early stopping patience: {PATIENCE}")
        print(f"  - 画像サイズ: {IMAGE_SIZE}×{IMAGE_SIZE}")
        print(f"  - 回転augmentation: 0-180度")
        print(f"  - カスタムaugmentation: ブラー・低解像度・ノイズ対策")
        print(f"  - Train images: {len(train_names)}")
        print(f"  - Val images: {len(val_names)}")
        
        # YOLOモデルをロード
        model = YOLO(YOLO_MODEL_NAME)
        
        # トレーニング実行
        results = model.train(
            data=str(yaml_file),
            epochs=EPOCHS,
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            patience=PATIENCE,
            project=str(OUTPUT_DIR),
            name='yolo11l-seg_3000mai',
            exist_ok=True,
            save=True,
            device=0 if torch.cuda.is_available() else 'cpu',
            degrees=180,  # 回転augmentation: 0-180度
            scale=0.5,    # スケールaugmentation
            augmentations=custom_transforms,  # カスタムAlbumentations
            mask_ratio=1,  # マスク解像度を画像サイズと同じに（滑らかな可視化のため）
        )
        
        print("\n✅ トレーニング完了")
        print(f"  モデル保存先: {OUTPUT_DIR / 'yolo11l-seg_3000mai' / 'weights' / 'best.pt'}")


✓ GPUが利用可能
  - GPU: NVIDIA GeForce RTX 3080 Ti Laptop GPU

📁 runsの保存場所: C:\Users\CorneAI\Eyelid_Iris_pupil_seg_comparison\YOLO11l-seg_3000mai\yolo11l-seg_3000mai
  - チェックポイント: C:\Users\CorneAI\Eyelid_Iris_pupil_seg_comparison\YOLO11l-seg_3000mai\yolo11l-seg_3000mai\weights
  - 学習曲線など: C:\Users\CorneAI\Eyelid_Iris_pupil_seg_comparison\YOLO11l-seg_3000mai\yolo11l-seg_3000mai

トレーニング設定:
  - モデル: yolo11l-seg.pt
  - エポック数: 100
  - バッチサイズ: 8
  - Early stopping patience: 20
  - 画像サイズ: 512×512
  - 回転augmentation: 0-180度
  - カスタムaugmentation: ブラー・低解像度・ノイズ対策
  - Train images: 2426
  - Val images: 574


C:\Users\CorneAI\AppData\Local\Temp\ipykernel_9480\5997615.py:49: UserWarning: Argument(s) 'scale_min, scale_max, interpolation' are not valid for transform Downscale
  A.Downscale(scale_min=0.25, scale_max=0.5, interpolation=cv2.INTER_LINEAR, p=1.0),
C:\Users\CorneAI\AppData\Local\Temp\ipykernel_9480\5997615.py:50: UserWarning: Argument(s) 'scale_min, scale_max, interpolation' are not valid for transform Downscale
  A.Downscale(scale_min=0.5, scale_max=0.75, interpolation=cv2.INTER_LINEAR, p=1.0),
C:\Users\CorneAI\AppData\Local\Temp\ipykernel_9480\5997615.py:55: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),  # ガウシアンノイズ
C:\Users\CorneAI\AppData\Local\Temp\ipykernel_9480\5997615.py:60: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=50, quality_upper=90, p=0.1),  # JPEG圧縮アーティファクト


New https://pypi.org/project/ultralytics/8.3.249 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.240  Python-3.13.5 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3080 Ti Laptop GPU, 16384MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, augmentations=[OneOf([
  MotionBlur(p=1.0, allow_shifted=True, angle_range=(0.0, 360.0), blur_limit=(3, 15), direction_range=(-1.0, 1.0)),
  GaussianBlur(p=1.0, blur_limit=(3, 11), sigma_limit=(0.5, 3.0)),
  MedianBlur(p=1.0, blur_limit=(3, 7)),
  Defocus(p=1.0, alias_blur=(0.1, 0.5), radius=(3, 10)),
], p=0.3), OneOf([
  Downscale(p=1.0, interpolation_pair={'upscale': 0, 'downscale': 0}, scale_range=(0.25, 0.25)),
  Downscale(p=1.0, interpolation_pair={'upscale': 0, 'downscale': 0}, scale_range=(0.25, 0.25)),
], p=0.2), OneOf([
  GaussNoise(p=1.0, mean_range=(0.0, 0.0), noise_scale_factor=1.0, per_channel=True, std_range=(0.2, 0.44)),
  ISONoise(p=1.0, color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
], p=0.15), Image

In [29]:
# ===== トレーニングの再開（Resume） =====

RUN_RESUME = False  # 再開する場合は True に設定

# RUN_RESUMEがFalseの場合は実行しない
if not RUN_RESUME:
    print("トレーニングを再開するには RUN_RESUME=True に設定してください。")
    if 'OUTPUT_DIR' in globals():
        print(f"\n💡 ヒント:")
        print(f"  - runsの保存場所: {OUTPUT_DIR / 'yolo11l-seg_3000mai'}")
        print(f"  - チェックポイント: {OUTPUT_DIR / 'yolo11l-seg_3000mai' / 'weights' / 'last.pt'}")
        print(f"  - 途中で止まった場合は、このセルで RUN_RESUME=True にして再実行してください")
    else:
        print(f"\n💡 ヒント:")
        print(f"  - runsの保存場所: YOLO11l-seg_3000mai/yolo11l-seg_3000mai")
        print(f"  - チェックポイント: YOLO11l-seg_3000mai/yolo11l-seg_3000mai/weights/last.pt")
        print(f"  - 途中で止まった場合は、このセルで RUN_RESUME=True にして再実行してください")
else:
    # 途中からスタートする場合に備えて、必要な変数を定義
    from pathlib import Path
    import cv2
    import albumentations as A

    if 'OUTPUT_DIR' not in globals():
        PROJECT_ROOT = Path('.').resolve()
        OUTPUT_DIR = PROJECT_ROOT / 'YOLO11l-seg_3000mai'

    if 'EPOCHS' not in globals():
        EPOCHS = 100
    if 'IMAGE_SIZE' not in globals():
        IMAGE_SIZE = 512
    if 'BATCH_SIZE' not in globals():
        BATCH_SIZE = 8
    if 'PATIENCE' not in globals():
        PATIENCE = 20

    # ===== カスタムAugmentation（ブラー・低解像度対策） =====
    custom_transforms = [
        # ブラー系augmentation（30%の確率で適用）
        A.OneOf([
            A.MotionBlur(blur_limit=(3, 15), p=1.0),      # モーションブラー
            A.GaussianBlur(blur_limit=(3, 11), p=1.0),    # ガウシアンブラー
            A.MedianBlur(blur_limit=7, p=1.0),            # メディアンブラー
            A.Defocus(radius=(3, 10), alias_blur=(0.1, 0.5), p=1.0),  # ピンボケ
        ], p=0.3),
        
        # 低解像度シミュレーション（20%の確率で適用）
        A.OneOf([
            A.Downscale(scale_min=0.25, scale_max=0.5, interpolation=cv2.INTER_LINEAR, p=1.0),
            A.Downscale(scale_min=0.5, scale_max=0.75, interpolation=cv2.INTER_LINEAR, p=1.0),
        ], p=0.2),
        
        # ノイズaugmentation（15%の確率で適用）
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),  # ガウシアンノイズ
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),  # ISOノイズ
        ], p=0.15),
        
        # 画質劣化augmentation（10%の確率で適用）
        A.ImageCompression(quality_lower=50, quality_upper=90, p=0.1),  # JPEG圧縮アーティファクト
        
        # コントラスト強調（低コントラスト画像対策）
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.1),
    ]

    import torch
    from ultralytics import YOLO
    
    # runsのディレクトリ
    RUNS_DIR = OUTPUT_DIR / 'yolo11l-seg_3000mai'
    LAST_CHECKPOINT = RUNS_DIR / 'weights' / 'last.pt'
    
    if not LAST_CHECKPOINT.exists():
        print(f"⚠ チェックポイントが見つかりません: {LAST_CHECKPOINT}")
        print(f"  最初からトレーニングを実行してください（前のセルで RUN_TRAIN=True）")
    else:
        print(f"✓ チェックポイントから再開します")
        print(f"  チェックポイント: {LAST_CHECKPOINT}")
        print(f"  保存場所: {RUNS_DIR.absolute()}")
        print(f"  カスタムaugmentation: ブラー・低解像度・ノイズ対策")
        
        # YOLOモデルをロード（チェックポイントから）
        model = YOLO(str(LAST_CHECKPOINT))
        
        # トレーニング再開（resume=Trueで最新のチェックポイントから自動的に再開）
        results = model.train(
            resume=True,  # これで最新のチェックポイントから自動的に再開
            epochs=EPOCHS,
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            patience=PATIENCE,
            project=str(OUTPUT_DIR),
            name='yolo11l-seg_3000mai',
            exist_ok=True,
            save=True,
            device=0 if torch.cuda.is_available() else 'cpu',
            degrees=180,
            scale=0.5,
            augmentations=custom_transforms,  # カスタムAlbumentations
            mask_ratio=1,  # マスク解像度を画像サイズと同じに（滑らかな可視化のため）
        )
        
        print("\n✅ トレーニング再開完了")
        print(f"  モデル保存先: {OUTPUT_DIR / 'yolo11l-seg_3000mai' / 'weights' / 'best.pt'}")


トレーニングを再開するには RUN_RESUME=True に設定してください。

💡 ヒント:
  - runsの保存場所: C:\Users\CorneAI\Eyelid_Iris_pupil_seg_comparison\YOLO11l-seg_3000mai\yolo11l-seg_3000mai
  - チェックポイント: C:\Users\CorneAI\Eyelid_Iris_pupil_seg_comparison\YOLO11l-seg_3000mai\yolo11l-seg_3000mai\weights\last.pt
  - 途中で止まった場合は、このセルで RUN_RESUME=True にして再実行してください


In [ ]:
# ===== 推論と可視化 =====

import matplotlib.pyplot as plt
import random
import cv2
import numpy as np
from ultralytics import YOLO

# 必要な変数が定義されていない場合のデフォルト値
if 'OUTPUT_DIR' not in globals():
    from pathlib import Path
    PROJECT_ROOT = Path('.').resolve()
    OUTPUT_DIR = PROJECT_ROOT / 'YOLO11l-seg_3000mai'

if 'IMAGE_SIZE' not in globals():
    IMAGE_SIZE = 512

if 'IMAGES_DIR' not in globals():
    if 'PROJECT_ROOT' not in globals():
        PROJECT_ROOT = Path('.').resolve()
    IMAGES_DIR = PROJECT_ROOT / 'Images' / 'images'

if 'CLASS_NAMES' not in globals():
    CLASS_NAMES = {
        0: 'conj',
        1: 'caruncle',
        2: 'iris_vis',
        3: 'iris_occ',
        4: 'pupil_vis',
        5: 'pupil_occ',
    }

RUN_INFER = True  # 推論を実行する場合は True に設定

# チェックポイントパス（常に同じパスを使用）
WEIGHTS_PATH = OUTPUT_DIR / 'yolo11l-seg_3000mai' / 'weights' / 'best.pt'

def ellipse_mask_from_full_max_contour(vis_255: np.ndarray, occ_255: np.ndarray):
    """可視部分と遮蔽部分から楕円マスクを生成（FullMax後処理）"""
    h, w = vis_255.shape
    vis_255 = (vis_255 > 0).astype(np.uint8) * 255
    occ_255 = (occ_255 > 0).astype(np.uint8) * 255
    full_255 = cv2.bitwise_or(vis_255, occ_255)
    m = (full_255 > 0).astype(np.uint8)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return np.zeros((h, w), dtype=np.uint8)
    cnt = max(cnts, key=cv2.contourArea)
    pts = cnt.reshape(-1, 2).astype(np.float32)
    if len(pts) < 5:
        return np.zeros((h, w), dtype=np.uint8)
    try:
        ellipse = cv2.fitEllipse(pts)
    except Exception:
        return np.zeros((h, w), dtype=np.uint8)
    mask = np.zeros((h, w), dtype=np.uint8)
    (cx, cy), (w_e, h_e), angle = ellipse
    center = (int(cx), int(cy))
    axes = (max(1, int(w_e / 2)), max(1, int(h_e / 2)))
    cv2.ellipse(mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
    return mask

def make_overlay(img_rgb, lid, iris, pupil, alpha=0.5):
    """画像にマスクをオーバーレイ"""
    h, w = img_rgb.shape[:2]
    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[lid > 0] = (255, 0, 0)  # 眼瞼: 赤
    color[iris > 0] = (0, 255, 0)  # 虹彩: 緑
    color[pupil > 0] = (0, 0, 255)  # 瞳孔: 青
    overlay = cv2.addWeighted(img_rgb, 1 - alpha, color, alpha, 0)
    return overlay

def visualize_samples(model, sample_names, title_prefix="YOLO11l-seg"):
    """サンプル画像の推論結果を可視化"""
    n = len(sample_names)
    fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for row, name in enumerate(sample_names):
        img_path = IMAGES_DIR / name
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(cv2.resize(img_bgr, (IMAGE_SIZE, IMAGE_SIZE)), cv2.COLOR_BGR2RGB)

        # YOLO推論
        results = model.predict(str(img_path), verbose=False, imgsz=IMAGE_SIZE, conf=0.25, retina_masks=True)

        # クラスごとのマスクを初期化
        class_masks = {cid: np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8) for cid in CLASS_NAMES.keys()}

        # 推論結果をパース
        if len(results) > 0 and results[0].masks is not None:
            masks = results[0].masks.data.cpu().numpy()  # (N, H, W)
            class_ids = results[0].boxes.cls.cpu().numpy().astype(int)  # (N,)

            for mask, cls_id in zip(masks, class_ids):
                if cls_id in class_masks:
                    # retina_masks=Trueの場合、マスクは元画像サイズなのでIMAGE_SIZEにリサイズ
                    if mask.shape != (IMAGE_SIZE, IMAGE_SIZE):
                        mask = cv2.resize(mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
                    # maskを0-255に変換
                    mask_255 = (mask > 0.5).astype(np.uint8) * 255
                    class_masks[cls_id] = np.maximum(class_masks[cls_id], mask_255)

        # 0-indexed: 0=conj, 1=caruncle, 2=iris_vis, 3=iris_occ, 4=pupil_vis, 5=pupil_occ
        conj = class_masks.get(0, np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8))
        caruncle = class_masks.get(1, np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8))
        iris_vis = class_masks.get(2, np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8))
        iris_occ = class_masks.get(3, np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8))
        pupil_vis = class_masks.get(4, np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8))
        pupil_occ = class_masks.get(5, np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8))

        # eyelid = conj + caruncle（可視化用）
        lid = ((conj > 0) | (caruncle > 0)).astype(np.uint8) * 255

        iris_raw = ((iris_vis > 0) | (iris_occ > 0)).astype(np.uint8) * 255
        pupil_raw = ((pupil_vis > 0) | (pupil_occ > 0)).astype(np.uint8) * 255

        iris_fullmax = ellipse_mask_from_full_max_contour(iris_vis, iris_occ)
        pupil_fullmax = ellipse_mask_from_full_max_contour(pupil_vis, pupil_occ)

        # Union mask: conj + iris_vis + pupil_vis の集合マスク
        union_vis = ((conj > 0) | (iris_vis > 0) | (pupil_vis > 0)).astype(np.uint8) * 255

        overlay_raw = make_overlay(img_rgb, lid, iris_raw, pupil_raw)
        overlay_fullmax = make_overlay(img_rgb, lid, iris_fullmax, pupil_fullmax)

        axes[row, 0].imshow(img_rgb)
        axes[row, 0].set_title(f"Original: {name}")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(overlay_raw)
        axes[row, 1].set_title("Raw (vis+occ)")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(overlay_fullmax)
        axes[row, 2].set_title("FullMax (ellipse)")
        axes[row, 2].axis("off")

        axes[row, 3].imshow(union_vis, cmap="gray", vmin=0, vmax=255)
        axes[row, 3].set_title("Union mask (conj + iris_vis + pupil_vis)")
        axes[row, 3].axis("off")

    plt.suptitle(f"{title_prefix} sample inference", y=0.99)
    plt.tight_layout()
    plt.show()

if RUN_INFER:
    if not WEIGHTS_PATH.exists():
        print(f"⚠ モデルが見つかりません: {WEIGHTS_PATH}")
        print("  先にトレーニングを実行してください（RUN_TRAIN=True）")
    else:
        print(f"モデルをロード中: {WEIGHTS_PATH}")
        model = YOLO(str(WEIGHTS_PATH))
        
        # サンプル画像を選択（validationセットからランダムに選ぶ）
        if 'val_names' in globals() and len(val_names) > 0:
            num_samples = min(3, len(val_names))
            sample_names = random.sample(val_names, k=num_samples)
            print(f"Selected {num_samples} random samples from validation set ({len(val_names)} images)")
        else:
            # val_namesが定義されていない場合は、画像ディレクトリから直接取得
            if IMAGES_DIR.exists():
                image_files = list(IMAGES_DIR.glob('*.jpg')) + list(IMAGES_DIR.glob('*.png'))
                if len(image_files) > 0:
                    num_samples = min(3, len(image_files))
                    selected_files = random.sample(image_files, k=num_samples)
                    sample_names = [f.name for f in selected_files]
                    print(f"Selected {num_samples} random samples from images directory ({len(image_files)} images)")
                else:
                    sample_names = []
                    print("Warning: No image files found in images directory")
            else:
                sample_names = []
                print(f"Warning: Images directory not found: {IMAGES_DIR}")
        
        if len(sample_names) > 0:
            print(f"Running inference on {len(sample_names)} samples...")
            visualize_samples(model, sample_names)
        else:
            print("推論するサンプル画像がありません")
else:
    print("推論を実行するには RUN_INFER=True に設定してください。")